In [1]:
import pandas as pd
import numpy as np

# --- Parameters ---
np.random.seed(42)

# Time period: past 5 years up to today
end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.DateOffset(years=5)
# Generate daily dates
date_index = pd.date_range(start=start_date, end=end_date, freq='D')

# Business info for synthetic data
store_name = "Sugar Cube Desserts"
store_id = "SCMCR001"
city = "Audenshaw"
region = "Greater Manchester"
country = "UK"

# Menu items and typical prices (based on public menu ranges)
menu_items = [
    dict(item="Krunch Cake", price=8.99),
    dict(item="Krunch Brownie", price=6.29),
    dict(item="Krunch Sundae", price=10.00),
    # Add more items if desired
    dict(item="Belgian Waffles", price=7.50),
    dict(item="Cookie Dough", price=5.50),
    dict(item="Doughnut", price=3.50),
    dict(item="Milkshake", price=4.99),
]

# Demographic buckets
age_groups = ["<18", "18-24", "25-34", "35-44", "45-54", "55+"]
genders = ["Female", "Male", "Non-binary/Other"]
income_segments = ["Low", "Lower-middle", "Middle", "Upper-middle", "High"]
regions = [
    "Audenshaw",
    "Manchester City Centre",
    "Stockport",
    "Salford",
    "Oldham",
    "Other Greater Manchester",
]

# Function to simulate number of transactions per day
def daily_txn_count(day_of_week):
    # More transactions on weekends
    base = 40
    if day_of_week >= 5:  # Saturday, Sunday
        return np.random.poisson(base * 1.5)
    else:
        return np.random.poisson(base)

# Cost assumptions: cost as a fraction of price
# e.g., average cost might be ~35-55% of price for dessert items; adjust for realism.
cost_pct_low, cost_pct_high = 0.35, 0.55

# --- Generate transaction-level data ---
records = []
for date in date_index:
    num_txns = daily_txn_count(date.dayofweek)
    for _ in range(num_txns):
        # Random menu item
        item = np.random.choice(menu_items)
        price = item["price"]

        # Simulate cost and profit margin
        cost_pct = np.random.uniform(cost_pct_low, cost_pct_high)
        cost = round(price * cost_pct, 2)
        profit = round(price - cost, 2)
        profit_margin = round(profit / price, 3)  # fraction

        # Demographic attributes for this transaction
        age = np.random.choice(age_groups, p=[0.1, 0.25, 0.3, 0.15, 0.1, 0.1])
        gender = np.random.choice(genders, p=[0.45, 0.45, 0.10])
        income = np.random.choice(income_segments, p=[0.2, 0.25, 0.3, 0.15, 0.1])
        # Region/city of customer
        cust_region = np.random.choice(regions, p=[0.25, 0.25, 0.15, 0.1, 0.1, 0.15])

        records.append({
            "transaction_id": f"{date.strftime('%Y%m%d')}-{np.random.randint(1, 99999):05d}",
            "date": date,
            "store_id": store_id,
            "store_name": store_name,
            "city": city,
            "region": region,
            "country": country,
            "item": item["item"],
            "price_gbp": price,
            "cost_gbp": cost,
            "profit_gbp": profit,
            "profit_margin": profit_margin,
            "age_group": age,
            "gender": gender,
            "income_segment": income,
            "customer_region": cust_region,
        })

df_txn = pd.DataFrame(records)

# Save transaction-level CSV
df_txn.to_csv("transaction_level.csv", index=False)

# --- Aggregated monthly summary ---
df_txn["year_month"] = df_txn["date"].dt.to_period("M")
agg_funcs = {
    "price_gbp": "sum",
    "cost_gbp": "sum",
    "profit_gbp": "sum",
    "transaction_id": "count"
}

df_monthly = df_txn.groupby("year_month").agg(agg_funcs).rename(
    columns={"price_gbp": "sales_gbp", "transaction_id": "num_transactions"}
).reset_index()

# Compute margin percentage
df_monthly["margin_pct"] = (df_monthly["profit_gbp"] / df_monthly["sales_gbp"]).round(3)

# Add demographic shares: example for income segments
income_dist = df_txn.groupby(["year_month", "income_segment"]).size().unstack(fill_value=0)
income_share = income_dist.div(income_dist.sum(axis=1), axis=0).add_prefix("share_income_")
df_monthly = df_monthly.join(income_share, on="year_month")

# Similarly for age groups
age_dist = df_txn.groupby(["year_month", "age_group"]).size().unstack(fill_value=0)
age_share = age_dist.div(age_dist.sum(axis=1), axis=0).add_prefix("share_age_")
df_monthly = df_monthly.join(age_share, on="year_month")

# Gender
gender_dist = df_txn.groupby(["year_month", "gender"]).size().unstack(fill_value=0)
gender_share = gender_dist.div(gender_dist.sum(axis=1), axis=0).add_prefix("share_gender_")
df_monthly = df_monthly.join(gender_share, on="year_month")

# Customer region/city
region_dist = df_txn.groupby(["year_month", "customer_region"]).size().unstack(fill_value=0)
region_share = region_dist.div(region_dist.sum(axis=1), axis=0).add_prefix("share_cust_region_")
df_monthly = df_monthly.join(region_share, on="year_month")

# Convert year_month back to string for CSV readability
df_monthly["year_month"] = df_monthly["year_month"].astype(str)

# Save aggregated summary CSV
df_monthly.to_csv("aggregated_summary.csv", index=False)

print("Generated transaction_level.csv and aggregated_summary.csv")

Generated transaction_level.csv and aggregated_summary.csv
